In [1]:
import pandas as pd

chexpert = pd.read_csv("mimic-cxr-2.0.0-chexpert.csv")
metadata = pd.read_csv("mimic-cxr-2.0.0-metadata.csv")
split    = pd.read_csv("mimic-cxr-2.0.0-split.csv")

print("chexpert:", chexpert.shape)
print("metadata:", metadata.shape)
print("split:   ", split.shape)
print("\nchexpert columns:", chexpert.columns.tolist())

chexpert: (227827, 16)
metadata: (377110, 12)
split:    (377110, 4)

chexpert columns: ['subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices']


### Filter

In [2]:
target_labels = ['Pneumonia', 'Pneumothorax']

chexpert_filtered = chexpert[['subject_id', 'study_id'] + target_labels].copy()

# Exclude rows with any uncertain label (-1)
has_uncertain = (chexpert_filtered[target_labels] == -1).any(axis=1)
chexpert_filtered = chexpert_filtered[~has_uncertain].copy()

# NaN → 0
chexpert_filtered[target_labels] = chexpert_filtered[target_labels].fillna(0)

print(f"Studies after excluding uncertain labels: {len(chexpert_filtered)}")
print("\nPositive count per label:")
print((chexpert_filtered[target_labels] == 1).sum())

Studies after excluding uncertain labels: 208462

Positive count per label:
Pneumonia       16509
Pneumothorax     9912
dtype: int64


In [3]:
has_positive = (chexpert_filtered[target_labels] == 1).any(axis=1)
all_negative = (chexpert_filtered[target_labels] == 0).all(axis=1)

selected = chexpert_filtered[has_positive | all_negative].copy()
print(f"Selected studies: {len(selected)}")

Selected studies: 208462


### Merge

In [4]:
df = selected.merge(split[['dicom_id', 'study_id', 'subject_id', 'split']],
                    on=['study_id', 'subject_id'], how='inner')

df = df.merge(metadata[['dicom_id', 'ViewPosition']],
              on='dicom_id', how='inner')

print(f"After merge: {len(df)} images")
print(df['ViewPosition'].value_counts())

After merge: 346467 images
AP                132756
PA                 89984
LATERAL            76128
LL                 32881
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP RLD                 2
SWIMMERS               1
AP LLD                 1
XTABLE LATERAL         1
PA RLD                 1
LPO                    1
Name: ViewPosition, dtype: int64


Only choose 1 image per study

In [5]:
df_pa = df.sort_values('dicom_id').groupby(['subject_id', 'study_id']).first().reset_index()

print(f"After merge: {len(df_pa)} images")
print(df_pa['ViewPosition'].value_counts())

After merge: 208462 images
AP                110011
PA                 41096
LATERAL            34327
LL                 16263
PA LLD                 2
RAO                    1
XTABLE LATERAL         1
AP RLD                 1
AP AXIAL               1
LPO                    1
Name: ViewPosition, dtype: int64


In [6]:
df_pa = df_pa[df_pa['ViewPosition'] == 'PA'].copy()

def build_path(row):
    p_prefix = f"p{str(row['subject_id'])[:2]}"
    return f"files/{p_prefix}/p{row['subject_id']}/s{row['study_id']}/{row['dicom_id']}.jpg"

df_pa['image_path'] = df_pa.apply(build_path, axis=1)

#Save Full
df_pa.to_csv("selected_dataset.csv", index=False)

#Download Path
df_pa['image_path'].to_csv("images_to_download.txt", index=False, header=False)

print(f"PA images total: {len(df_pa)}")
print(f"\nSplit distribution:")
print(df_pa['split'].value_counts())
print(df_pa.head(3))

PA images total: 41096

Split distribution:
train       40364
test          411
validate      321
Name: split, dtype: int64
    subject_id  study_id  Pneumonia  Pneumothorax  \
0     10000032  50414267        0.0           0.0   
1     10000032  53189527        0.0           0.0   
11    10000980  50985099        0.0           0.0   

                                        dicom_id  split ViewPosition  \
0   02aa804e-bde0afdd-112c0b34-7bc16630-4e384014  train           PA   
1   2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab  train           PA   
11  6ad03ed1-97ee17ee-9cf8b320-f7011003-cd93b42d  train           PA   

                                           image_path  
0   files/p10/p10000032/s50414267/02aa804e-bde0afd...  
1   files/p10/p10000032/s53189527/2a2277a9-b0ded15...  
11  files/p10/p10000980/s50985099/6ad03ed1-97ee17e...  


### Check

In [7]:
print("Positive label count (PA only):")
print((df_pa[target_labels] == 1).sum())
print(f"\nTotal: {len(df_pa)} images")
print(f"At least one positive label: {(df_pa[target_labels] == 1).any(axis=1).sum()}")
print(f"All negative: {(df_pa[target_labels] == 0).all(axis=1).sum()}")

Positive label count (PA only):
Pneumonia       2647
Pneumothorax    1086
dtype: int64

Total: 41096 images
At least one positive label: 3710
All negative: 37386


### Balancing Sample

In [9]:
train_df = df_pa[df_pa['split'] == 'train'].copy()
test_df  = df_pa[df_pa['split'] == 'test'].copy()
val_df   = df_pa[df_pa['split'] == 'validate'].copy()

# balancing sample
positive_train = train_df[(train_df[target_labels] == 1).any(axis=1)]
negative_train = train_df[(train_df[target_labels] == 0).all(axis=1)]

neg_sample_train = negative_train.sample(
    n=len(positive_train) * 2, 
    random_state=42
)

train_balanced = pd.concat([positive_train, neg_sample_train]) \
                   .sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train balanced: {len(train_balanced)} (pos {len(positive_train)}, neg {len(neg_sample_train)})")
print(f"Test original:  {len(test_df)}")
print(f"Val original:   {len(val_df)}")

Train balanced: 10866 (pos 3622, neg 7244)
Test original:  411
Val original:   321


In [10]:
# final dataset
df_final = pd.concat([train_balanced, test_df, val_df], ignore_index=True)

df_final.to_csv("selected_dataset_final.csv", index=False)
df_final['image_path'].to_csv("images_to_download_final.txt", index=False, header=False)

### Select Small Portion

In [11]:
# Choose only 10%
df_small = train_balanced.sample(frac=0.1, random_state=42).reset_index(drop=True)

print(f"Small sample total: {len(df_small)} images")
print(f"\nPositive label count:")
print((df_small[target_labels] == 1).sum())

# Save
df_small.to_csv("selected_dataset_small.csv", index=False)
df_small['image_path'].to_csv("images_to_download_small.txt", index=False, header=False)
print("\nSaved: selected_dataset_small.csv")

Small sample total: 1087 images

Positive label count:
Pneumonia       283
Pneumothorax    100
dtype: int64

Saved: selected_dataset_small.csv


In [11]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/opt/apps/intel19/python3/3.9.7/bin/python3.9 -m pip install --upgrade pip' command.


### Download Images

In [ ]:
import requests, os, concurrent.futures
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()


session = requests.Session()
login_url = "https://physionet.org/login/"
r = session.get(login_url)
csrf = r.cookies.get("csrftoken")
login_data = {
    "username": os.getenv("PHYSIONET_USERNAME"),
    "password": os.getenv("PHYSIONET_PASSWORD"),
    "csrfmiddlewaretoken": csrf
}
session.post(login_url, data=login_data, headers={"Referer": login_url})


base_url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
save_dir = "images"
os.makedirs(save_dir, exist_ok=True)

def download_one(path):
    local_path = os.path.join(save_dir, path)
    if os.path.exists(local_path):
        return None
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    try:
        r = session.get(base_url + path, timeout=30)
        if r.status_code == 200:
            with open(local_path, 'wb') as f:
                f.write(r.content)
            return None
        else:
            return (path, r.status_code)
    except Exception as e:
        return (path, str(e))


with open("images_to_download_small.txt") as f:
    paths = [l.strip() for l in f if l.strip()]

failed = []
with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_one, p): p for p in paths}
    for fut in tqdm(concurrent.futures.as_completed(futures), total=len(paths)):
        result = fut.result()
        if result:
            failed.append(result)

print(f"Done! Failed: {len(failed)}")


if failed:
    with open("failed_downloads.txt", "w") as f:
        for path, err in failed:
            f.write(f"{path}\t{err}\n")

100%|██████████| 1087/1087 [19:12<00:00,  1.06s/it]

Done! Failed: 0


### Train Model

In [12]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from sklearn.model_selection import train_test_split

Set Up

In [13]:
CSV_PATH = "selected_dataset_small.csv"   
IMAGE_ROOT = "images"
BATCH_SIZE = 32
NUM_EPOCHS = 15
PATIENCE = 5
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "checkpoints_small"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TARGET_LABELS = ['Pneumonia', 'Pneumothorax']  

Class

In [14]:
class MIMICCXRDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.labels = TARGET_LABELS
        
        self.df['full_image_path'] = self.df['image_path'].apply(
            lambda x: os.path.join(IMAGE_ROOT, x)
        )
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row[self.labels].values.astype(np.float32))
        return image, label

Transform

In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [16]:
torch.multiprocessing.set_start_method('spawn', force=True)

In [17]:
dataset = MIMICCXRDataset(CSV_PATH, transform=transform)

# small dataset all train，cut 80/20 -> validation
train_idx, val_idx = train_test_split(range(len(dataset)), test_size=0.2, random_state=42, stratify=dataset.df[TARGET_LABELS].values.sum(axis=1))

train_dataset = torch.utils.data.Subset(dataset, train_idx)
val_dataset   = torch.utils.data.Subset(dataset, val_idx)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=False,     
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=False
)

print(f"Small dataset → Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Small dataset → Train: 869 | Val: 218


In [18]:
# model
model = models.densenet121(weights='DEFAULT')
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, len(TARGET_LABELS))
model = model.to(DEVICE)

# Loss, Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                 factor=0.5, patience=2)

Evaluate

In [19]:
def evaluate(loader, model, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    
    aucs = [roc_auc_score(all_labels[:, i], all_preds[:, i]) for i in range(len(TARGET_LABELS))]
    return np.mean(aucs), aucs

In [ ]:
# Training
best_val_auc = 0.0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    
    train_loss /= len(train_dataset)
    
    # Validation
    val_auc, val_aucs = evaluate(val_loader, model, DEVICE)
    scheduler.step(val_auc)
    
    print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Mean AUC: {val_auc:.4f} | "
          f"AUCs: Pneumonia={val_aucs[0]:.4f}, Pneumothorax={val_aucs[1]:.4f}")
    
    # Checkpoint & Early Stopping
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_small_model.pth"))
        
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            break

print(f"\n Completed! Val Mean AUC = {best_val_auc:.4f}")

Epoch 1/15:  96%|█████████▋| 27/28 [05:02<00:11, 11.13s/it][W410 03:06:59.729294457 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.757118786 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.786193990 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.818332807 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.854148540 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.892160746 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.933402956 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.952538181 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.961334168 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:06:59.970308670 NNPACK.cpp:56] Could not 

Epoch  1 | Train Loss: 0.5550 | Val Mean AUC: 0.6793 | AUCs: Pneumonia=0.5634, Pneumothorax=0.7953


Epoch 2/15:  96%|█████████▋| 27/28 [04:54<00:10, 10.88s/it][W410 03:12:28.342812021 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.369364830 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.398473406 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.430504088 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.465433214 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.504148491 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.545545853 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.564623310 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.573115164 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:12:28.582170027 NNPACK.cpp:56] Could not 

Epoch  2 | Train Loss: 0.3314 | Val Mean AUC: 0.7154 | AUCs: Pneumonia=0.5890, Pneumothorax=0.8418


Epoch 3/15:  96%|█████████▋| 27/28 [04:54<00:10, 10.95s/it][W410 03:17:56.554998594 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.581426958 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.610329405 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.642953663 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.678052777 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.716692833 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.757792053 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.776837889 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.785312700 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:17:56.794277825 NNPACK.cpp:56] Could not 

...

Epoch 4/15:  96%|█████████▋| 27/28 [04:54<00:10, 10.87s/it][W410 03:23:23.794401611 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:23.820849762 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:23.849859412 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:23.882651005 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:23.917594728 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:23.953010788 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:24.993594069 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:24.012893162 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:24.021249772 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:23:24.030275259 NNPACK.cpp:56] Could not 

Epoch  4 | Train Loss: 0.0963 | Val Mean AUC: 0.7242 | AUCs: Pneumonia=0.6114, Pneumothorax=0.8371


Epoch 5/15:  96%|█████████▋| 27/28 [04:54<00:10, 10.91s/it][W410 03:28:51.812483249 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:51.836813186 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:51.863485870 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:51.893504528 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:51.925827750 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:51.960891538 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:52.998495764 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:52.016257196 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:52.024026652 NNPACK.cpp:56] Could not initialize NNPACK! Reason: Unsupported hardware.
[W410 03:28:52.032306216 NNPACK.cpp:56] Could not 

Epoch  5 | Train Loss: 0.0592 | Val Mean AUC: 0.6969 | AUCs: Pneumonia=0.5751, Pneumothorax=0.8186


Epoch 6/15:  32%|███▏      | 9/28 [01:38<03:28, 10.96s/it]

Epoch  1 | Train Loss: 0.5550 | Val Mean AUC: 0.6793 | AUCs: Pneumonia=0.5634, Pneumothorax=0.7953<br>
Epoch  2 | Train Loss: 0.3314 | Val Mean AUC: 0.7154 | AUCs: Pneumonia=0.5890, Pneumothorax=0.8418<br>
Epoch  3 | Train Loss: 0.2011 | Val Mean AUC: 0.7275 | AUCs: Pneumonia=0.6284, Pneumothorax=0.8265<br>
Epoch  4 | Train Loss: 0.0963 | Val Mean AUC: 0.7242 | AUCs: Pneumonia=0.6114, Pneumothorax=0.8371<br>
Epoch  5 | Train Loss: 0.0592 | Val Mean AUC: 0.6969 | AUCs: Pneumonia=0.5751, Pneumothorax=0.8186<br>
...